In [7]:
import re
import json
import pandas as pd

def obtener_medias_spark(ruta_html):
    print(f"Procesando el archivo: {ruta_html}...\n")
    
    # 1. Leer el archivo HTML
    try:
        with open(ruta_html, "r", encoding="utf-8") as f:
            html = f.read()
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{ruta_html}'.")
        return

    # 2. Función para aislar y convertir los arrays de JS a diccionarios de Python
    def extraer_variable(var_name):
        # Buscar el bloque de la variable solicitada
        match = re.search(rf'var {var_name} = (\[.*?\]);', html, re.DOTALL)
        if not match: 
            return None
        
        # El HTML de Spark (especialmente v5) tiene claves sin comillas (ej. x: "15:13...").
        # Usamos esta expresión regular para añadir comillas y hacerlo un JSON válido.
        json_str = re.sub(r'([{,]\s*)([a-zA-Z0-9_]+)\s*:', r'\1"\2":', match.group(1))
        return json.loads(json_str)

    # 3. Diccionario con la correspondencia de variables
    metricas = {
        'Input Rate (records/sec)': 'v1',
        'Process Rate (records/sec)': 'v2',
        'Input Rows (records/batch)': 'v3',
        'Batch Duration (ms)': 'v4'
    }

    print("--- MEDIAS GLOBALES ---")
    for nombre, var_name in metricas.items():
        datos = extraer_variable(var_name)
        if datos:
            df = pd.DataFrame(datos)
            # Las métricas principales guardan el valor en la clave 'y'
            media = df['y'].mean()
            print(f"{nombre}: {media:.2f}")
        else:
            print(f"{nombre}: No se encontraron datos.")

    print("\n--- DESGLOSE DE OPERATION DURATION (ms) ---")
    datos_v5 = extraer_variable('v5')
    if datos_v5:
        df_v5 = pd.DataFrame(datos_v5)
        # Eliminamos la columna de tiempo 'x' y convertimos los valores a números (float)
        df_v5 = df_v5.drop(columns=['x']).astype(float)
        
        # Calculamos la media por columna y la imprimimos
        medias_v5 = df_v5.mean().round(2)
        for operacion, media in medias_v5.items():
            print(f"{operacion}: {media}")
    else:
        print("No se encontraron los datos de las operaciones internas (v5).")

# ==========================================
# CÓMO EJECUTAR EL SCRIPT
# ==========================================
# 1. Guarda el código fuente que me pasaste en un archivo llamado 'spark_ui.html'
# 2. Asegúrate de tener pandas instalado (pip install pandas)
# 3. Llama a la función:

if __name__ == "__main__":
    # Cambia "spark_ui.html" por la ruta real de tu archivo si está en otra carpeta
    obtener_medias_spark("Prueba_datosVolsiVolno.html")

Procesando el archivo: Prueba_datosVolsiVolno.html...

--- MEDIAS GLOBALES ---
Input Rate (records/sec): 361.59
Process Rate (records/sec): 391.04
Input Rows (records/batch): 525.79
Batch Duration (ms): 1122.62

--- DESGLOSE DE OPERATION DURATION (ms) ---
addBatch: 527.93
commitOffsets: 260.42
getBatch: 0.18
latestOffset: 5.03
queryPlanning: 16.63
walCommit: 310.21
